# 03 - Daily_Km_Assigned regression
Merges the notebook's two regression pipelines (cell 9: simple MLP; cell 12: "ultra complex" MLP) into `src.prediction.train_regressor`.

Results recorded in the **original** notebook (test set, real data, see `docs/experiments.md`): simple MLP best R2 0.3615 (standard scaler),
ultra-complex best R2 0.2660 (robust). This notebook recomputes with the refactored, leak-free pipeline, so numbers will differ; none are quoted here.

Needs PyTorch.

In [ ]:
import logging
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():          # notebooks/ -> repo root
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

import pandas as pd
from src import config as cfg
from src.data.loader import load_raw_dataset, resolve_data_path

# Real data if data/raw/kochi_induction_dataset.csv exists, else the SYNTHETIC sample.
USE_SAMPLE = not cfg.RAW_DATA_PATH.exists()
path, SOURCE = resolve_data_path(use_sample=USE_SAMPLE)
df = load_raw_dataset(path)
print("data source:", SOURCE, "| shape:", df.shape)

In [ ]:
import dataclasses
from src.prediction.train_regressor import train_regressor

simple_cfg = cfg.REGRESSOR_PRESETS["simple"]
regressor, table, report = train_regressor(df, simple_cfg, report_dir=cfg.DEFAULT_RUN_DIR, data_source=SOURCE)
table

The scaler was chosen on **validation** RMSE. Test columns are reported afterwards and were not used to choose.

In [ ]:
from src.evaluation.visualization import plot_training_history
plot_training_history(report["history"], "Simple regressor", "MSE loss");

## The larger architecture (notebook cell 12)
Slower; it was the worse performer in the original run.

In [ ]:
big_cfg = dataclasses.replace(cfg.REGRESSOR_PRESETS["ultra_complex"], scalers=("standard",))   # one scaler to save time
_, table_big, _ = train_regressor(df, big_cfg, save=False, data_source=SOURCE)
table_big

## The regressor trains on all rows, Standby included
Like the notebook, this regressor sees rows with `Daily_Km_Assigned == 0` (Standby). Whether to train only on Service rows was never tested; see `docs/experiments.md`.